In [ ]:
import matplotlib.pylab as plt 
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge

# 1. Cost functions

For $N$ samples and $p$ features, linear regression has the form

$$
\mathbf{y} \approx f(\mathbf{X}) = \mathbf{X}\mathbf{w}_1 + w_0\mathbf{1}_N,
$$

where $\mathbf{X}\in\mathbb{R}^{N\times p}$, $\mathbf{w}_1\in\mathbb{R}^{p}$ contains the feature weights, and $w_0\in\mathbb{R}$ is the intercept.

#### 1.1 What are the rows of $\mathbf{X}$?

Each row corresponds to one data point (observation).

With $N$ observations and $p$ input features,

$$
\mathbf{X} =
\begin{bmatrix}
- & \mathbf{x}_1^{\mathsf T} & - \\
- & \mathbf{x}_2^{\mathsf T} & - \\
& \vdots & \\
- & \mathbf{x}_N^{\mathsf T} & -
\end{bmatrix}
\in \mathbb{R}^{N\times p}.
$$

Row $i$ contains the $p$ feature values of observation $i$.

#### 1.2 What are the columns of $\mathbf{X}$?

Each column corresponds to one feature (input variable).

For example, with two features,

$$
\mathbf{X} =
\begin{bmatrix}
x_{11} & x_{12} \\
x_{21} & x_{22} \\
\vdots & \vdots \\
x_{N1} & x_{N2}
\end{bmatrix}.
$$

The first column contains feature 1 for every observation, and the second contains feature 2.

#### 1.3 How does $\tilde{\mathbf{X}}$ look like in this case?

$$
\mathbf{y} \approx f(\mathbf{X}) = \mathbf{X}\mathbf{w}_1 + w_0\mathbf{1}_N \longrightarrow \mathbf{y} \approx \tilde{\mathbf{X}}\mathbf{w}
$$

We add a column of ones to include the intercept $w_0$:

$$
\tilde{\mathbf{X}} =
\begin{bmatrix}
\mathbf{x}_1^{\mathsf T} & 1 \\
\mathbf{x}_2^{\mathsf T} & 1 \\
\vdots & \vdots \\
\mathbf{x}_N^{\mathsf T} & 1
\end{bmatrix},
\qquad
\mathbf{w} =
\begin{bmatrix}
\mathbf{w}_1 \\
w_0
\end{bmatrix}.
$$

The shape changes from $\mathbf{X} \in \mathbb{R}^{N\times p}$ to $\tilde{\mathbf{X}} \in \mathbb{R}^{N\times (p + 1)}$.

For each observation, the residual/error is the difference between the observed and predicted values:

$$
\epsilon_i = y_i-f(\mathbf{x}_i).
$$

Collecting all residuals into a vector gives

$$
\boldsymbol{\epsilon} =
\begin{pmatrix}
y_1-f(\mathbf{x}_1) \\
\vdots \\
y_N-f(\mathbf{x}_N)
\end{pmatrix}
= \mathbf{y}-\tilde{\mathbf{X}}\mathbf{w}.
$$

Two common cost functions are the mean-squared error (MSE) and mean-absolute error (MAE):

$$
\begin{aligned}
\mathrm{MSE} &= \frac{1}{N}\sum_{i=1}^{N}\epsilon_i^2
= \frac{1}{N}\sum_{i=1}^{N}\left(y_i-f(\mathbf{x}_i)\right)^2, \\
\mathrm{MAE} &= \frac{1}{N}\sum_{i=1}^{N}|\epsilon_i|
= \frac{1}{N}\sum_{i=1}^{N}\left|y_i-f(\mathbf{x}_i)\right|.
\end{aligned}
$$

#### 1.4 In the Jupyter notebook, write a Python function that computes these two cost functions given an error term $\boldsymbol{\epsilon} = \mathbf{y} - \tilde{\mathbf{X}}\mathbf{w}$

In [ ]:
def mean_squared_error(error_vector):
    """Return the mean of the squared residuals."""
    errors = np.asarray(error_vector, dtype=float)
    return np.mean(errors ** 2)

In [ ]:
def mean_absolute_error(error_vector):
    """Return the mean of the absolute residuals."""
    errors = np.asarray(error_vector, dtype=float)
    return np.mean(np.abs(errors))

Examples for both cost functions:

In [ ]:
for errors in ([0, 0, 0], [1, 1, 1], [-2, 0, 2]):
    print(f"Errors: {errors}, MSE: {mean_squared_error(errors):.3f}, MAE: {mean_absolute_error(errors):.3f}")

#### 1.5 What is the shape of these cost functions as a function of the error?

For one residual ($N=1$),

$$
\mathrm{MSE}(\epsilon)=\epsilon^2,
\qquad
\mathrm{MAE}(\epsilon)=|\epsilon|.
$$

MSE is a smooth parabola. MAE is V-shaped, with a sharp bend at $\epsilon=0$. Both functions are convex, symmetric about zero, and have their minimum value $0$ at $\epsilon=0$.

In [ ]:
x_axis = np.linspace(-3, 3, 501)
y_mse = [mean_squared_error(error) for error in x_axis]
y_mae = [mean_absolute_error(error) for error in x_axis]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_axis, y_mse, label=r"MSE: $\epsilon^2$")
ax.plot(x_axis, y_mae, label=r"MAE: $|\epsilon|$")
ax.set(xlabel=r"Error $\epsilon$", ylabel="Cost", title="Cost functions for one residual")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

#### 1.6  Are both loss functions differentiable for all $\boldsymbol{\epsilon}$? What implications does this have for gradient based optimization like gradient descent?

MSE is differentiable $\forall\epsilon_i$, since

$$
\frac{\partial\mathrm{MSE}}{\partial\epsilon_i}=\frac{2\epsilon_i}{N}.
$$

MAE is not differentiable when $\epsilon_i=0$. For $\epsilon_i \neq 0$,

$$
\frac{\partial\mathrm{MAE}}{\partial\epsilon_i}=
\begin{cases}
-1/N, & \epsilon_i<0, \\
+1/N, & \epsilon_i>0.
\end{cases}
$$

Gradient descent updates the weights $\mathbf{w}$. Since

$$
\boldsymbol{\epsilon}=\mathbf{y}-\tilde{\mathbf{X}}\mathbf{w},
$$

$$
\frac{\partial\boldsymbol{\epsilon}}{\partial\mathbf{w}^{\mathsf T}}
=-\tilde{\mathbf{X}}.
$$

These residual derivatives are related to the weight gradients through the chain rule.

$$
\nabla_{\mathbf{w}}\mathrm{MSE}
=\left(\frac{\partial\boldsymbol{\epsilon}}{\partial\mathbf{w}^{\mathsf T}}\right)^{\mathsf T}
\nabla_{\boldsymbol{\epsilon}}\mathrm{MSE}
=-\tilde{\mathbf{X}}^{\mathsf T}\left(\frac{2}{N}\boldsymbol{\epsilon}\right)
=-\frac{2}{N}\tilde{\mathbf{X}}^{\mathsf T}\boldsymbol{\epsilon}.
$$

MSE has a gradient everywhere, so standard gradient descent works. MAE is not differentiable at $\epsilon_i=0$, so standard gradient descent needs a way to handle this point. Instead, we can use subgradient descent, which generalises the gradient to non-smooth functions. Also, unlike for MSE, the MAE gradient does not get smaller near the minimum, so with a fixed learning rate gradient descent oscillates around it and the learning rate has to be decreased.

#### 1.7 Which loss function is more sensitive to outliers and why?

MSE is more sensitive to outliers with large residuals, because it penalizes errors quadratically, whereas MAE penalizes them linearly. For each error, the contributions to the cost scale as

$$
\text{MSE}\propto\epsilon_i^2,
\qquad
\text{MAE}\propto|\epsilon_i|.
$$

The MSE slope magnitude, $2|\epsilon_i|/N$, also grows with the error, while the MAE slope magnitude is $1/N$ for nonzero errors. 

MSE can pull the fitted model towards outliers, making it fit the other points worse. MAE is less affected because it does not square the errors.

# 2. Regularization

Assume that the columns of $\mathbf{X}$ are linearly independent.
As a refresher of linear algebra, recall when the linear system $\mathbf{X}\mathbf{w} = \mathbf{y}$ has

#### 2.1 One unique solution

Let $\mathbf{X}$ be an $N \times p$ matrix ($N$ samples, $p$ unknowns) and $[\mathbf{X}|\mathbf{y}]$ the augmented matrix. The system has a unique solution when $\text{rank}(\mathbf{X}) = \text{rank}([\mathbf{X}|\mathbf{y}]) = p$, which means $\mathbf{y}$ lies in the column space of $\mathbf{X}$ and there are no free variables. Since the columns of $\mathbf{X}$ are linearly independent, $\text{rank}(\mathbf{X}) = p$ already holds (this needs $N \geq p$). If $\mathbf{X}$ is square ($N = p$), it is invertible and $\mathbf{w} = \mathbf{X}^{-1}\mathbf{y}$ for any $\mathbf{y}$.

#### 2.2 No solution

There is no solution when $\text{rank}(\mathbf{X}) < \text{rank}([\mathbf{X}|\mathbf{y}])$. Then $\mathbf{y}$ is not in the column space of $\mathbf{X}$, so no combination of the columns gives $\mathbf{y}$. This is what usually happens in regression: we have more samples than parameters ($N > p$) and the data is noisy, so instead of solving the system exactly we minimise the error.

#### 2.3 An infinite number of solutions

There are infinitely many solutions when $\text{rank}(\mathbf{X}) = \text{rank}([\mathbf{X}|\mathbf{y}]) < p$. In this case $\mathbf{X}$ has a non-trivial null space, so if $\mathbf{w}^*$ is a solution, $\mathbf{w}^* + \mathbf{z}$ is also one for any $\mathbf{z}$ with $\mathbf{X}\mathbf{z} = \mathbf{0}$ (there are $p - \text{rank}(\mathbf{X})$ free parameters). This always happens when there are more columns than rows ($p > N$) and the system is consistent, because $\text{rank}(\mathbf{X}) \leq N < p$. In this case the columns can't be linearly independent.

#### 2.4 Give a geometrical interpretation of the matrix Rank (10 bonus point if you use the Manim package)

The rank is the maximum number of linearly independent columns of $\mathbf{X}$, i.e. the dimension of the space they span. If we look at $\mathbf{X}$ as a linear map $\mathbf{w} \mapsto \mathbf{X}\mathbf{w}$ from $\mathbb{R}^p$ to $\mathbb{R}^N$, the rank is the dimension of the set of all possible outputs $\mathbf{X}\mathbf{w}$: with rank 1 everything is mapped onto a line, with rank 2 onto a plane, and so on. If $\text{rank}(\mathbf{X}) < p$, some directions (the null space) are squashed to zero, so the map loses information and cannot be inverted (for a square matrix this means $\det \mathbf{X} = 0$). This is also why $\mathbf{X}\mathbf{w} = \mathbf{y}$ only has a solution if $\mathbf{y}$ lies in this subspace, and why the solution is not unique when some directions are lost.

**Bonus (Manim):** the animation below shows this for $2 \times 2$ matrices. A rank-2 matrix maps the grid onto the whole plane and can be undone. A rank-1 matrix (second column $= 2 \times$ first column) squashes the plane onto a line: the null space (yellow) is mapped to zero, and a point $\mathbf{y}$ that is not on the line cannot be reached. The code is in `rank_animation.py` (run `manim -qm rank_animation.py RankInterpretation`).

![Rank animation](rank_animation.gif)

#### 2.5 In general, why can't we solve the linear equation using $\mathbf{y} = \tilde{\mathbf{X}}^{-1}\textbf{w}$? (1 point)

There are two problems with this expression. First, it is not the right rearrangement: from $\mathbf{y} = \tilde{\mathbf{X}}\mathbf{w}$ we would get $\mathbf{w} = \tilde{\mathbf{X}}^{-1}\mathbf{y}$, not $\mathbf{y} = \tilde{\mathbf{X}}^{-1}\mathbf{w}$. Also, $\mathbf{y}$ is the data we already know and $\mathbf{w}$ is what we want to find, so this equation does not help us. Second, $\tilde{\mathbf{X}}$ is $N \times (p+1)$ and usually not square ($N \gg p+1$), so $\tilde{\mathbf{X}}^{-1}$ does not exist. Even a square $\tilde{\mathbf{X}}$ is singular if its columns are linearly dependent.

On top of that, with noisy data there is usually no exact solution (see 2.2), so we minimise $\|\mathbf{y} - \tilde{\mathbf{X}}\mathbf{w}\|_2^2$ instead.

#### 2.6 Differentiate above formula step by step and show what will we have?

We differentiate the least-squares objective with respect to the weight vector $\mathbf{w}$:

$$
\mathcal{L}(\mathbf{w}) = (\mathbf{y}-\tilde{\mathbf{X}}\mathbf{w})^\top(\mathbf{y}-\tilde{\mathbf{X}}\mathbf{w})
$$

First, we expand the formula using matrix product rules as well as transposed matrix product rule ($(\mathbf{A}\mathbf{B})^\top = \mathbf{B}^\top\mathbf{A}^\top$):

$$
\mathcal{L}(\mathbf{w}) = \mathbf{y}^\top\mathbf{y} - \mathbf{y}^\top\tilde{\mathbf{X}}\mathbf{w} - \mathbf{w}^\top\tilde{\mathbf{X}}^\top\mathbf{y} + \mathbf{w}^\top\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{w}
$$

We can then observe that the two middle terms are scalars  ($1\times1$), and a scalar equals its transpose:
$\mathbf{y}^\top\tilde{\mathbf{X}}\mathbf{w} = (\mathbf{y}^\top\tilde{\mathbf{X}}\mathbf{w})^\top = \mathbf{w}^\top\tilde{\mathbf{X}}^\top\mathbf{y}$. Therefore

$$
\mathcal{L}(\mathbf{w}) = \mathbf{y}^\top\mathbf{y} - 2\,\mathbf{w}^\top\tilde{\mathbf{X}}^\top\mathbf{y} + \mathbf{w}^\top\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{w}
$$

To differentiate this equation some matrix calculus formulas are applied:

| Term | Rule | Derivative |
|---|---|---|
| $\mathbf{y}^\top\mathbf{y}$ | does not depend on $\mathbf{w}$ | $\mathbf{0}$ |
| $-2\,\mathbf{w}^\top\mathbf{a}$, with $\mathbf{a}=\tilde{\mathbf{X}}^\top\mathbf{y}$ | $\frac{\partial}{\partial\mathbf{w}}(\mathbf{w}^\top\mathbf{a}) = \mathbf{a}$ | $-2\,\tilde{\mathbf{X}}^\top\mathbf{y}$ |
| $\mathbf{w}^\top\mathbf{A}\mathbf{w}$, with $\mathbf{A}=\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}$ | $\frac{\partial}{\partial\mathbf{w}}(\mathbf{w}^\top\mathbf{A}\mathbf{w}) = (\mathbf{A}+\mathbf{A}^\top)\mathbf{w} = 2\mathbf{A}\mathbf{w}$ (since $\mathbf{A}$ is symmetric) | $2\,\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{w}$ |

This yields:

$$
\boxed{\frac{\partial\mathcal{L}}{\partial\mathbf{w}} = -2\,\tilde{\mathbf{X}}^\top\mathbf{y} + 2\,\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{w} = -2\,\tilde{\mathbf{X}}^\top(\mathbf{y}-\tilde{\mathbf{X}}\mathbf{w})}
$$

Which is a $(p+1)\times1$ vector. 

#### 2.7 if we want $\| y - \tilde{X} w \|_2^2$ to be minimum, what should the derivative be equal to?

In order to minimize this function its derivative should be zero:
$$
\frac{\partial\mathcal{L}}{\partial\mathbf{w}}= -2\,\tilde{\mathbf{X}}^\top(\mathbf{y} - \tilde{\mathbf{X}}\mathbf{w}) = \mathbf{0}
$$

Which gives:
$$
\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\,\mathbf{w} = \tilde{\mathbf{X}}^\top\mathbf{y}
$$


These are the normal equations. If $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}$ is invertible (i.e. the columns of $\tilde{\mathbf{X}}$ are linearly independent), the solution is

$$
\boxed{\mathbf{w}^* = (\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^\top\mathbf{y}}
$$

The reason why this is a minimum and not a maximum: the Hessian is $\nabla^2_{\mathbf{w}}\mathcal{L} = 2\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}$. For any $\mathbf{v}$ we have $\mathbf{v}^\top\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{v} = \|\tilde{\mathbf{X}}\mathbf{v}\|^2 \ge 0$, so the Hessian is positive semi-definite, which means $\mathcal{L}$ is convex and this point is a minimum.


#### 2.8 What is the Hat matrix and what does its diagonal values correspond to?

Insert $\mathbf{w}^*$ into the model to get the fitted values:

$$
\hat{\mathbf{y}} = \tilde{\mathbf{X}}\mathbf{w}^* = \underbrace{\tilde{\mathbf{X}}(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^\top}_{\mathbf{H}}\,\mathbf{y} = \mathbf{H}\mathbf{y}
$$

The $N \times N$ matrix $\mathbf{H}$ is called the Hat matrix. The Hat matrix projects a vector of observations onto a vector of predicted  values by the regression.

The diagonal elements $h_{ii}$ are called leverages:

$$
h_{ii} = \tilde{\mathbf{x}}_i^\top(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{x}}_i = \frac{\partial \hat{y}_i}{\partial y_i}
$$

- $h_{ii}$ is the leverage of sample $i$: how strongly its own observed value $y_i$ influences its own prediction $\hat{y}_i$.
- It depends only on the inputs $\tilde{\mathbf{x}}_i$, not on $y_i$. It measures how far sample $i$ lies from the centre of the training data in feature space (it is closely related to the Mahalanobis distance from the centroid).
- $0 \le h_{ii} \le 1$ (with an intercept, $\frac{1}{N} \le h_{ii} \le 1$), and the average leverage is $\bar{h} = (p+1)/N$.

#### 2.9 What is Willson’s plot and how does it help in outlier detection?

The Williams plot shows the standardized residuals of every sample (y-axis) as a function of its leverage $h_{ii}$. Standardized residuals are prediction errors $y_i - \hat{y}_i$ divided by the model's typical error, so it is expressed in standard deviation's units.  Threshold lines are usually drawn at $\pm 3$ standard deviations (horizontal) and at the warning leverage $h^* = 3(p+1)/N$ .

It is useful because it detects two different kinds of outliers in one picture: points above or below the horizontal lines are response outliers (badly predicted $y$), while points to the right of $h^*$ are structural outliers (unusual descriptors, far from the training data). Points beyond both limits are the most dangerous, since they can strongly distort the fitted model. For new samples, $h > h^*$ means the model is extrapolating, so the prediction should not be trusted.

#### 2.10 What happens if some columns are linearly dependent? What is the connection to feature selection?

If a column is a linear combination of others  then $\operatorname{rank}(\tilde{\mathbf{X}}) < p+1$. Compared to the augmented matrix, the system $\tilde{\mathbf{X}}\mathbf{w} = \mathbf{y}$ then has no exact solution if $\operatorname{rank}(\tilde{\mathbf{X}}) < \operatorname{rank}([\tilde{\mathbf{X}}|\mathbf{y}])$, and infinitely many if the two ranks are equal (see 2.2 and 2.3). As a result:

- There is a non-zero $\mathbf{v}$ with $\tilde{\mathbf{X}}\mathbf{v} = \mathbf{0}$, so $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{v} = \mathbf{0}$ as well. This means that $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}$ is a singular matrix ($\det = 0$) and is thus non-invertible. Therefore $(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}$ does not exist, and the Hat matrix can't be computed with this formula.
- The normal equations have infinitely many solutions: if $\mathbf{w}^*$ is a solution, so is $\mathbf{w}^* + c\,\mathbf{v}$ for any $c$, and every one gives exactly the same predictions $\tilde{\mathbf{X}}\mathbf{w}$. The predictions may still be defined, but the coefficients are not identifiable, so they cannot be interpreted as "the effect of feature $j$".

From the standpoint of feature selection, features containing linear dependencies carry redundant information. Feature selection removes the redundancy, for example by:
- dropping one feature of each highly correlated pair (correlation filter, variance inflation factor $\mathrm{VIF}_j = 1/(1 - R_j^2)$),
- projecting onto uncorrelated directions (PCA / PLS), which strictly is feature extraction rather than selection,
- using LASSO ($L_1$) regularisation, which tends to keep one feature of a correlated group and set the others exactly to zero.

This restores full column rank, so $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}$ becomes invertible and well-conditioned. It gives stable, interpretable coefficients, fewer parameters and less overfitting. The other route is to keep all features and **regularise** (ridge), which is what the next questions are about.

#### 2.11 What will be the new cost function after adding the regularization term?

With regularisation we add a penalty on the size of the weights and the complexity of the system:

$$
\boxed{\mathcal{L}(\mathbf{w}) = \|\mathbf{y} - \tilde{\mathbf{X}}\mathbf{w}\|_2^2 + \lambda\|\mathbf{w}\|_2^2 = (\mathbf{y} - \tilde{\mathbf{X}}\mathbf{w})^\top(\mathbf{y} - \tilde{\mathbf{X}}\mathbf{w}) + \lambda\,\mathbf{w}^\top\mathbf{w}}, \qquad \lambda \ge 0
$$

The first term rewards fitting the data. The second term penalises large coefficients, which keeps the model smooth, less sensitive to noise and most importantly keeps the system generalised and not fixated on the training data. It also makes $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I}$ invertible for any $\lambda > 0$, even when the columns are linearly dependent (see 2.13).


#### 2.12 What will be the new 𝑤 when we differentiate the new cost function and set it to zero.

Expand as in 2.6:

$$
\mathcal{L} = \mathbf{y}^\top\mathbf{y} - 2\mathbf{w}^\top\tilde{\mathbf{X}}^\top\mathbf{y} + \mathbf{w}^\top\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{w} + \lambda\,\mathbf{w}^\top\mathbf{I}\,\mathbf{w}
$$

Differentiate term by term, using $\frac{\partial}{\partial\mathbf{w}}(\lambda\mathbf{w}^\top\mathbf{w}) = 2\lambda\mathbf{w}$:

$$
\nabla_{\mathbf{w}}\mathcal{L} = -2\tilde{\mathbf{X}}^\top\mathbf{y} + 2\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{w} + 2\lambda\mathbf{w}
$$

Setting it to zero gives us:

$$
\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{w} + \lambda\mathbf{I}\mathbf{w} = \tilde{\mathbf{X}}^\top\mathbf{y}
\;\;\Longrightarrow\;\;
(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I})\,\mathbf{w} = \tilde{\mathbf{X}}^\top\mathbf{y}
$$

$$
\boxed{\mathbf{w} = (\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I})^{-1}\tilde{\mathbf{X}}^\top\mathbf{y}}
$$

The Hessian $2(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I})$ is positive definite for $\lambda > 0$ (see 2.13), so this is the unique global minimum.

#### 2.13 Prove that the part of Hat matrix where we want to take the inverse from is always reversible after we introduce the regularization term.

Inserting the new $\mathbf{w}$ from 2.12 into $\hat{\mathbf{y}} = \tilde{\mathbf{X}}\mathbf{w}$, the Hat matrix becomes $\mathbf{H} = \tilde{\mathbf{X}}(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I})^{-1}\tilde{\mathbf{X}}^\top$, so the part we have to invert is $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I}$. Let $\mathbf{A} = \tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I}$  and $\lambda > 0$ an arbitrary regularisation parameter. In order to show that $\mathbf{A}$ is invertible, we show that $\mathbf{A}\mathbf{v} = \mathbf{0}$ only has the solution $\mathbf{v} = \mathbf{0}$. 


We suppose a vector  $\mathbf{v}$ satisfies

$$
(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I})\,\mathbf{v} = \mathbf{0}
$$

Multiplying both sides on the left by $\mathbf{v}^\top$ gives:

$$
\mathbf{v}^\top\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}\mathbf{v} + \lambda\,\mathbf{v}^\top\mathbf{v} = 0
$$

Using $\mathbf{v}^\top\tilde{\mathbf{X}}^\top = (\tilde{\mathbf{X}}\mathbf{v})^\top$, both terms are squared lengths of vectors:

$$
\underbrace{\|\tilde{\mathbf{X}}\mathbf{v}\|^2}_{\ge\,0} + \lambda\,\underbrace{\|\mathbf{v}\|^2}_{\ge\,0} = 0
$$

A sum of two non-negative numbers can only be zero if both are zero. Since $\lambda > 0$, this forces $\|\mathbf{v}\|^2 = 0$, i.e. $\mathbf{v} = \mathbf{0}$.

Hence $\mathbf{A}\mathbf{v} = \mathbf{0}$ has only the trivial solution, so the columns of $\mathbf{A}$ are linearly independent, the determinant is non-zero and hence  $\mathbf{A}$ is invertible, for any $\tilde{\mathbf{X}}$.

What does $\lambda$ contribute to this proof?  With $\lambda = 0$ the same argument only gives $\|\tilde{\mathbf{X}}\mathbf{v}\|^2 = 0$, i.e. $\tilde{\mathbf{X}}\mathbf{v} = \mathbf{0}$. If the columns of $\tilde{\mathbf{X}}$ are linearly dependent, this has non-zero solutions $\mathbf{v}$, so $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}}$ is not invertible (2.10). The term $\lambda\|\mathbf{v}\|^2$ is what rules this out.
Therefore, for any $\lambda > 0$, $\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}} + \lambda\mathbf{I}$ is always reversible.



#### 2.14  What is the shape of the parabola as a function of $a$?

In [ ]:
def parabola(x, a = 0.5): 
    return a * x ** 2

In [ ]:
x_axis_parabola = np.linspace(-10, 10, 100)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=300) #just to have a nicer image
a_values = [-1, 0, 0.5, 1, 2]
colors = plt.cm.magma(np.linspace(0.15, 0.85, len(a_values))) #because i love magma palette
for a, color in zip(a_values, colors):
    plt.plot(x_axis_parabola, parabola(x_axis_parabola, a), color=color, label=f"a = {a}")
plt.legend()

Increasing $|a|$ makes the parabola narrower, and vice versa; with positive $a$, parabola points upwards, with negative - downwards; $a = 0$ gives a straight line. So a smaller $|a|$ gives a flatter, smoother function, and since regularisation penalises large coefficients like $a$, it keeps the fitted function smooth and helps against overfitting. 

#### 2.15 Plot the approximation to the function for different order polynomials ($N \in \{1, 2, 16\}$) and with different regularization strength ($\lambda \in \{0, 10^{-3}, 10^{-2}, 1\}$). What do you observe 

In [ ]:
def true_function(X):
    return np.cos(1.5 * np.pi * X)

In [ ]:
X_test = np.linspace(0, 1, 100) # some grid for us on the x axis

In [ ]:
n_samples = 10 # the number of points we will sample from true_function
degrees = [1, 2, 16] # the polynomial degrees we will test

np.random.seed(7) # fixed seed, so the plots match our description
X = np.sort(np.random.rand(n_samples))
y = true_function(X) + np.random.randn(n_samples) * 0.1 # add some scaled random noise

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=300) #just to have a nicer image
plt.scatter(X, y,color=plt.cm.magma(0.55), label='noisy samples')
plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.25),  label='true function')
plt.legend()

The following code will fit a polynomial regression, you need to fill the degree

In [ ]:
polynomial_features = PolynomialFeatures(degree=2,
                                             include_bias=False)
linear_regression = LinearRegression()
pipeline = Pipeline([("polynomial_features", polynomial_features),
                     ("linear_regression", linear_regression)])
pipeline.fit(X[:, np.newaxis], y)

To plot the result, you can use the following code

In [ ]:
#plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="model of degree ")
#plt.scatter(X, y, label='noisy samples')
#plt.plot(X_test, true_function(X_test), c='r', label='true function')
#plt.legend()

In [ ]:
# let's combine it in one cell:

fig, ax = plt.subplots(figsize=(8, 5),dpi=300) #just to have a nice picture
colors = plt.cm.magma(np.linspace(0.35, 0.85, len(degrees))) #and of course we'll use our favourite magma 

for degree, color in zip(degrees,colors) :
    polynomial_features = PolynomialFeatures(degree=degree,
                                             include_bias=False)
    linear_regression = LinearRegression()
    pipeline = Pipeline([("polynomial_features", polynomial_features),
                        ("linear_regression", linear_regression)])
    pipeline.fit(X[:, np.newaxis], y)

    plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]),
             color=color,
             label=f"model of degree {degree}")

plt.scatter(X, y, s=20, color=plt.cm.magma(0.15), label="noisy samples")
plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.15), label="true function")
plt.ylim(-3, 3) #degree 16 explodes at the edges, so we zoom in
plt.legend(fontsize="small")
plt.show()
    

Here, we see that the model of degree 1 approximates the true function rather badly, because it cannot capture the real curvature of the function. The models of degree 2 and 16, however, can capture this curvature. But the model of degree 16 goes almost exactly through every noisy point and oscillates strongly between them, especially near the edges (it has more coefficients than we have data points), so it is not smooth at all. In order to choose one model we can follow the Occam's razor: we choose the simplest one which gives the acceptable result - here, this would be the model of degree 2, as degree 16 overfits this small dataset. 

Next, we can investigate the effect of the regularization parameter $\lambda$ (function parameter `alpha`), For this, you can use the following code 

In [ ]:
polynomial_features = PolynomialFeatures(degree=1,
                                             include_bias=False)
ridge_regression = Ridge(alpha=0.001)
pipeline_ridge = Pipeline([("polynomial_features", polynomial_features),
                     ("ridge_regression", ridge_regression)])
pipeline_ridge.fit(X[:, np.newaxis], y)


For plotting you can reuse the following code

In [ ]:
#plt.plot(X_test, pipeline_ridge.predict(X_test[:, np.newaxis]), label="model of degree with regularization strength")
#plt.scatter(X, y, label='noisy samples')
#plt.plot(X_test, true_function(X_test), c='r', label='true function')
#plt.legend()

In [ ]:
#let's combine everything again
degrees = [1, 2, 16]
for degree in degrees: 
    fig, ax = plt.subplots(figsize=(8, 5),dpi=300) #just to have a nice picture
    alphas = [0, 0.001, 0.01, 1]
    colors = plt.cm.magma(np.linspace(0.35, 0.85, len(alphas))) #and of course we'll use our favourite magma 
    for alpha, color in zip(alphas,colors) :
        polynomial_features = PolynomialFeatures(degree=degree,
                                                include_bias=False)
        ridge_regression = LinearRegression() if alpha == 0 else Ridge(alpha=alpha) #changed this because with 10 samples, degree 16 Ridge(alpha=0) gives an ill-conditioned matrix warning
        pipeline_ridge = Pipeline([("polynomial_features", polynomial_features),
                            ("ridge_regression", ridge_regression)])
        pipeline_ridge.fit(X[:, np.newaxis], y)

        plt.plot(X_test, pipeline_ridge.predict(X_test[:, np.newaxis]),
                color=color,
                label=f"model of degree {degree} with regularization strength {alpha}")

    plt.scatter(X, y, s=20, color=plt.cm.magma(0.15), label="noisy samples")
    plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.15), label="true function")
    plt.ylim(-3, 3) #same zoom as above
    plt.legend(fontsize="small")
    plt.show()
    


Here, the same principle applies: all models with the degree of 1 perform badly whatever $\lambda$ is, because a straight line is too simple. For degree 2, a small $\lambda$ barely changes the fit. For degree 16, the regularisation makes a big difference. Without it the model overfits, while $\lambda = 10^{-3}$ or $10^{-2}$ gives a much smoother curve close to the true function. This is because the wiggly fit needs very large coefficients, which the regularisation term penalises, so the function becomes smoother (like a flatter parabola in 2.14). The Occam's razor can be followed here with respect to the regularisation strength: restricting the magnitude in coefficients reduces the model's complexity without changing its polynomial degree. However, one should keep in mind that excessive regularisation can produce underfitting. With $\lambda = 1$ all models become too flat, even for degree 2 and 16.

#### 2.16 What do you observe if you change the number of samples from the function?

In [ ]:
#let's change the number of samples
n_samples = 100 # the number of points we will sample from true_function
degrees = [1, 2, 16] # the polynomial degrees we will test

X = np.sort(np.random.rand(n_samples))
y = true_function(X) + np.random.randn(n_samples) * 0.1 # add some scaled random noise

degrees = [1, 2, 16]
for degree in degrees: 
    fig, ax = plt.subplots(figsize=(8, 5),dpi=300) #just to have a nice picture
    alphas = [0, 0.001, 0.01, 1]
    colors = plt.cm.magma(np.linspace(0.35, 0.85, len(alphas))) #and of course we'll use our favourite magma 
    for alpha, color in zip(alphas,colors) :
        polynomial_features = PolynomialFeatures(degree=degree,
                                                include_bias=False)
        ridge_regression = LinearRegression() if alpha == 0 else Ridge(alpha=alpha) #changed this because with 10 samples, degree 16 Ridge(alpha=0) gives an ill-conditioned matrix warning
        pipeline_ridge = Pipeline([("polynomial_features", polynomial_features),
                            ("ridge_regression", ridge_regression)])
        pipeline_ridge.fit(X[:, np.newaxis], y)

        plt.plot(X_test, pipeline_ridge.predict(X_test[:, np.newaxis]),
                color=color,
                label=f"model of degree {degree} with regularization strength {alpha}")

    plt.scatter(X, y, s=10, color=plt.cm.magma(0.15), label="noisy samples")
    plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.25), label="true function")
    plt.legend()
    plt.show()
    



Generally, the more samples we have, the better we can fit the polynomial. Also, the polynomials of high degrees, which were overfitting quite strongly with a low number of datapoints, perform significantly better. With 100 points and only 17 coefficients, the degree 16 model can no longer go through every noisy point, so even without regularisation it follows the true function well. The only exception is the very edge ($x \approx 1$), where there are few samples and the curve shoots off. Regularisation also matters less now: the degree 16 curves for different $\lambda$ are closer together (except $\lambda = 1$, which is still too flat), because the error term sums over many more points while the penalty stays the same. On the other hand, the models of degree 1 and 2 look almost the same as before. They are too simple for this function, so their error does not come from overfitting but from the model being too inflexible (bias). For example, the degree 2 curve cannot reach the depth of the minimum, no matter how many points we use. So more data and regularisation both help against overfitting (variance), but neither can fix a model that is too simple. 

#### 2.17 Why do we need a test set in machine learning?

Test set is needed to estimate how good the model's performance is on unseen data. It can show whether the model generalises well or whether it has just "learnt by heart" the training set and overfits. The training error is not enough for this, because it is too optimistic: the model was fitted to exactly these points (e.g. the degree 16 model in 2.15 goes almost through every training point). The test set gives an unbiased estimate only if it is never used for training or for choosing the model.

#### 2.18 If we need to optimize hyperparameters, do we use the test set to select the best hyperparameters? (1 point)

No, to optimise the hyperparameters we usually use the validation set (or cross-validation on the training data). The test set is used only for the final evaluation, because otherwise the evaluation would be too optimistic: choosing hyperparameters on the test set is data leakage, since information from the test data gets into the model.  